# Docling PDF extractor

En este notebook se muestra cómo usar Docling para extraer texto de uno o varios PDF.

**Fuentes:**

- [Docling: An Efficient Open-Source Toolkit for AI-driven Document Conversion](https://arxiv.org/pdf/2501.17887)
- [Docling GitHub Repository](https://github.com/docling-project/docling?tab=readme-ov-file)
- [Docling Documentation](https://docling-project.github.io/docling/)
- [Docling Web](https://www.docling.ai)
- [Docling Examples](https://docling-project.github.io/docling/examples/)
- [Convert Any Document To LLM Knowledge with Docling (YouTube)](https://www.youtube.com/watch?v=YAgjtZKLVKo)
- [Docling: Get your documents ready for gen AI](https://www.youtube.com/watch?v=xUxhqJLVgRE)
- [IBM Docling Free and OpenSource Advanced Document Parser](https://www.youtube.com/watch?v=EeMy1Qv9rPY)
- [Docling: Get Your Documents Ready for Gen AI](https://www.youtube.com/watch?v=03Wh8s5bUwc)
- [Technical Workshop: Meet Docling: The “Pandas” for Document AI](https://www.youtube.com/watch?v=ehONq7mY81Y)
- [Docling Workshops Code](https://github.com/docling-project/docling-workshops)

## Conceptos básicos

Un objeto `DoclingDocument` es como un “árbol” (o un JSON grande) que describe qué hay en el PDF (o el tipo de documento que sea) y dónde está cada cosa.

### 1) Documento

En el nivel más alto tenemos el documento y dentro de él lo que hay es una colección de items (elementos) que se dividen en dos categorías principales:

* **Items de contenido**
* **Items de estructura**

Los **items de contenido** son los elementos que representan el contenido principal del documento y pueden ser:

* `texts`. Todos los elementos que tienen una representación textual (párrafo, encabezado de sección, ecuación, etc.). La clase base es `TextItem`.
* `tables`. Todas las tablas, de clase `TableItem`. Puede llevar anotaciones de estructura.
* `figures`. Todas las imágenes, de clase `PictureItem`. Puede llevar anotaciones de estructura.
* `key_value_items`. Todos los elementos clave-valor.

Los **items de estructura** son los que representan la organización del documento, como:

* `body`. El nodo raíz de una estructura de árbol para el cuerpo principal del documento.
* `furniture`. El nodo raíz de una estructura de árbol para todos los elementos que no pertenecen al cuerpo (encabezados, pies de página, etc.).
* `groups`. Un conjunto de elementos que no representan contenido, sino que actúan como contenedores para otros elementos de contenido (por ejemplo, una lista o un capítulo). En la práctica se reconoce porque tiene `children`: referencias a otros items. Un grupo puede tener como children una mezcla de items distintos: `TextItem`, `TableItem`, `FigureItem` e incluso otros grupos.

### 2) Jerarquía: `parent` y `children`

Docling suele representar la estructura con referencias:

* `parent`: a qué bloque pertenece este item
* `children`: qué items cuelgan de este item

Así se construye la “estructura lógica” del documento (lo que sería el orden de lectura).

### 3) Procedencia (`prov`, de provenance)

Informa de dónde se encuantra cada elemenento en el PDF y esto es lo que hace a Docling potente. En muchos items veremos algo como:

* `page_no`: número de página
* `bbox`: un rectángulo (coordenadas) que marca dónde está el item en la página
* `charspan`: si es un texto, qué rango de caracteres del texto original cubre

Sirve para trazabilidad, resaltar cosas en el PDF, debugging, etc.

### 4) Etiquetas (labels)

Cada item lleva una etiqueta que indica su “rol”:

* `page_header`, `page_footer`
* `paragraph`, `title`, `section_header`
* `table`
* `figure`
* etc.

Docling luego usa esas labels para exportar a Markdown/HTML de forma razonable.

### 5) Tablas e imágenes

* **Tabla**: normalmente un `TableItem` que a su vez contiene su estructura interna (filas/celdas).
* **Figura/imagen**: item con `prov` (dónde está) y metadatos (si se extrajo la imagen, el caption, etc., según la configuración del conversor).

### 6) Furniture

En Docling, `furniture` son los elementos de maquetación que no forman parte del contenido principal del documento.

Suele incluir cosas como:

* encabezados (`page_header`)
* pies de página (`page_footer`)
* números de página
* marcos, líneas, separadores
* logos repetidos, watermarks

La idea es separar:

* **Contenido**: el texto y las imágenes “reales” con información útil (párrafos, títulos, tablas, etc.)
* **Furniture**: lo repetitivo o decorativo que aparece por diseño y normalmente no queremos en el texto limpio o en los embeddings.

Si queremos filtrarlo en el pipeline, normalmente podemos:

* ignorar items con `content_layer == FURNITURE`
* ignorar labels concretas como `page_header` o `page_footer`

## Setup

Comprobamos que Docling y sus dependencias están correctamente instalados.

In [ ]:
import sys
import torch

print(f"{sys.version=}")
print(f"{torch.__version__=}")

In [ ]:
import time
import rich
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import display, Markdown
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode, TableStructureOptions
from docling_core.transforms.serializer.markdown import MarkdownDocSerializer
from docling_core.types.doc import DoclingDocument, ContentLayer, DocItemLabel
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

In [ ]:
# Procesamos un PDF de ejemplo
source = "https://hutchesonlab.fiu.edu/wp-content/uploads/sample-pdf.pdf"  # ruta al documento en local o URL
converter = DocumentConverter()
doc = converter.convert(source).document
print(doc.export_to_markdown())

In [ ]:
PATH_ARTIFACTS = Path.home() / ".cache" / "docling" / "models"  # Docling models are stored here
PATH_ARTIFACTS.exists()

In [ ]:
PATH_DATA = Path("..") / "data"
PATH_INPUT = PATH_DATA / "raw"
PATH_OUTPUT = PATH_DATA / "interim" / "docling"

PATH_OUTPUT.mkdir(exist_ok=True, parents=True)

## Convertir PDF a texto en Markdown

In [ ]:
PAGE_BREAK_PLACEHOLDER = "<!-- page_break -->"


def create_pdf_pipeline_options() -> PdfPipelineOptions:
    return PdfPipelineOptions(
        enable_remote_services=True,
        do_ocr=False,
        do_table_structure=True,
        table_structure_options=TableStructureOptions(
            mode=TableFormerMode.ACCURATE,
        ),
    )


converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=create_pdf_pipeline_options(),
            backend=PyPdfiumDocumentBackend,
        )
    }
)

In [ ]:
# PDFs a procesar
pdf_files = sorted(PATH_INPUT.glob("*.pdf"))
pdf_files

In [ ]:
tm_total_start = time.perf_counter()

with tqdm(pdf_files, total=len(pdf_files), unit="pdf", desc="Converting PDFs") as pbar:
    for pdf_path in pbar:
        tm_start = time.perf_counter()
        try:
            result = converter.convert(pdf_path)
            print(f"Calidad de conversión: {result.confidence.mean_grade}")
            doc = result.document

            md_content = doc.export_to_markdown(
                page_break_placeholder=PAGE_BREAK_PLACEHOLDER,
                include_annotations=True,
                mark_annotations=True,
            )

            # Guardamos el Markdown y el JSON resultante
            out_path = PATH_OUTPUT / f"{pdf_path.stem}.md"
            out_path.write_text(md_content, encoding="utf-8")
            doc.save_as_json(PATH_OUTPUT / f"{pdf_path.name[:-4]}.json")

        except Exception as e:
            pbar.write(f"FAIL {pdf_path.name}: {e}")

        finally:
            tm_end = time.perf_counter()
            pbar.write(f"OK  {pdf_path.name}  ({tm_end - tm_start:.2f}s)")

tm_total_end = time.perf_counter()
print(f"All conversions completed in {tm_total_end - tm_total_start:.2f} seconds.")

## Inspección del objeto `DoclingDocument`

`DoclingDocument` es una estructura de datos de Pydantic que establece una interfaz estructurada para representar el documento y sus elementos (tablas, imágenes, secciones, etc.).

In [ ]:
# Seleccionamos un documento para inspeccionar su contenido
FILE_NAME = pdf_files[0].stem
FILE_NAME

In [ ]:
# Importamos su JSON y reconstruimos el DoclingDocument
json_path = PATH_OUTPUT / f"{FILE_NAME}.json"
doc = DoclingDocument.load_from_json(json_path)

type(doc)

El objeto `DoclingDocument` tiene diversos atributos y métodos disponibles.

In [ ]:
# dir(doc)

In [ ]:
doc.model_dump().keys()

In [ ]:
# Metadata del documento
print("Tipo del documento:", doc.origin.mimetype)
print("Número de páginas del documento:", doc.num_pages())
print("Nombre del documento:", doc.name)
print("Número de grupos en el documento:", len(doc.groups))
print("Número de textos en el documento:", len(doc.texts))
print("Número de imágenes en el documento:", len(doc.pictures))
print("Número de tablas en el documento:", len(doc.tables))
print("Total items en el documento:", len(list(doc.iterate_items())))

In [ ]:
# Cada objeto DoclingDocument contiene una serie de content items, que son los elementos extraídos del PDF.
print(f"{type(doc.texts)=}")
print(f"{type(doc.tables)=}")
print(f"{type(doc.pictures)=}")
print(f"{type(doc.key_value_items)=}")

In [ ]:
doc.pages.keys()

In [ ]:
# Imprime las características de la primera página
print(doc.pages[1])

Los grupos de contenido (*content groups*) son estructuras que agrupan diferentes *content items* que están relacionados entre sí, por ejemplo los elementos de una lista. Podemos acceder a ellos a través del atributo `groups` del `DoclingDocument`.

In [ ]:
len(doc.groups)

In [ ]:
# Extramos el grupo 2 y encontramos a sus hijos
my_group = doc.groups[2]
refs = my_group.children

# Recorremos los elementos hijos e
# imprimimos el texto original
for x in refs:
    text = x.resolve(doc=doc)
    print(text.orig)

In [ ]:
# Veamos algunas propiedades de estos textos
for x in refs:
    text = x.resolve(doc=doc)
    print(text.self_ref)
    print(text.enumerated)
    print(text.marker)
    print(text.text)

In [ ]:
# Si queremos saber el parent de un content item
my_text = doc.texts[10]
parent_ref = my_text.parent

# Resolver la referencia principal
parent_item = parent_ref.resolve(doc=doc)
rich.print(parent_item)

In [ ]:
# Extrae el grupo 30 y encuentra a sus hijos
my_group = doc.groups[30]
refs = my_group.children

# Recorre los elementos secundarios, determina
# su ubicación e imprime el texto original
for x in refs:
    text = x.resolve(doc=doc)
    print(text.orig)

In [ ]:
# Si queremos saber el parent de un content item
my_text = doc.texts[228]
parent_ref = my_text.parent

# Resolver la referencia principal
parent_item = parent_ref.resolve(doc=doc)
rich.print(parent_item)

In [ ]:
rich.print(doc.texts[228])

In [ ]:
my_group = doc.groups[30]
serializer = MarkdownDocSerializer(doc=doc)
ser_res = serializer.serialize(item=my_group)
print(ser_res.text)

Vemos que ha convertido el `"marker": "\u25cf"` a su equivalente en Markdown (`-`).

### Textos

In [ ]:
print(f"Nº de fragmentos de textos en el documento: {len(doc.texts)}")

In [ ]:
type(doc.texts[11])

In [ ]:
doc.texts[11].text

In [ ]:
rich.print(doc.texts[11])

In [ ]:
# Veamos ahora un header de sección
rich.print(doc.texts[22])

### Tablas

In [ ]:
print(f"Nº de tablas en el documento: {len(doc.tables)}")

In [ ]:
# rich.print(doc.tables[0])

In [ ]:
# Podemos exportar la tabla a Markdown
doc.tables[1].export_to_markdown(doc=doc)

In [ ]:
display(Markdown(doc.tables[1].export_to_markdown(doc=doc)))

In [ ]:
# Podemos exportar la tabla a otros formatos, como DataFrame de pandas
doc.tables[1].export_to_dataframe(doc=doc)

### Pictures

In [ ]:
print(f"Nº de imágenes en el documento: {len(doc.pictures)}")

Docling detecta que en nuestro documentos hay elementos de tipo `PictureItem`, que representan ítems de imágenes. Pero dado que en las opciones del pipeline hemos indicado que no queremos extraer imágenes, en el Markdown resultante no se incluyen.

Sin embargo, los items de tipo `PictureItem` siguen existiendo en el objeto `DoclingDocument`, aunque sin su contenido (es decir, sin la imagen). Esto es útil porque nos permite identificar qué partes del documento original correspondían a imágenes con `PAGE_BREAK_PLACEHOLDER = "<!-- page_break -->"`, aunque no tengamos el contenido de esas imágenes.

In [ ]:
type(doc.pictures[0])

In [ ]:
# Si queremos obtener la imagen, todas van a ser None
type(doc.pictures[0].get_image(doc=doc))

In [ ]:
rich.print(doc.pictures[0])

In [ ]:
doc.pictures[0].caption_text(doc=doc)

## Eliminar elementos no deseados

### Eliminar imágenes

Imaginemos que queremos eliminar los elementos de tipo `PictureItem`, que representan ítems de imágenes y que en nuestro caso sabemos que no son relevantes para el análisis. Para ello, podemos usar el método `delete_items()` del `DoclingDocument`, pasando una lista de referencias a los items que queremos eliminar.

In [ ]:
total_pics = len(doc.pictures)
print(f"Nº de imágenes en el documento: {total_pics}")
total_texts = len(doc.texts)
print(f"Nº de textos en el documento: {total_texts}")

In [ ]:
# Eliminamos todas las imágenes del documento
doc.delete_items(node_items=doc.pictures)

total_pics_after_deletion = len(doc.pictures)
print(f"Nº de imágenes después de la eliminación: {total_pics_after_deletion}")
total_texts_after_deletion = len(doc.texts)
print(f"Nº de textos después de la eliminación: {total_texts_after_deletion}")

In [ ]:
rich.print(doc.texts[1])

Vemos que al eliminar las imágenes, el texto relacionado con ellas también ha desaparecido. Esto se debe a que estos textos tenían una referencia `parent` a la imagen, y al eliminar la imagen, el sistema de referencias de Docling también elimina los textos relacionados para mantener la coherencia del documento.

### Eliminar furniture

Ahora vamos a analizar los ítems que ha sido marcados como `furniture` para ver qué contienen y si efectivamente son elementos de maquetación que no queremos incluir en el análisis.

In [ ]:
# Vamos a ver los textos que son furniture
# No forman parte del contenido principal del documento
for item in doc.texts:
    if item.content_layer == ContentLayer.FURNITURE:
        print(item.self_ref, item.text, item.label)

Vemos que efectivamente los textos que han sido marcados como `furniture` corresponden a elementos de maquetación como encabezados, pies de página, números de página, etc., que no forman parte del contenido principal del documento. Por lo tanto, si queremos eliminar estos elementos para quedarnos solo con el contenido relevante, podemos usar el mismo método `delete_items()` pasando las referencias a estos ítems de `furniture`.

In [ ]:
total_texts = len(doc.texts)
print(f"Nº de textos en el documento: {total_texts}")

In [ ]:
items_to_delete = [item for item in doc.texts if item.content_layer == ContentLayer.FURNITURE]
doc.delete_items(node_items=items_to_delete)

In [ ]:
total_texts_after_deletion = len(doc.texts)
print(f"Nº de textos en el documento: {total_texts_after_deletion}")

Ahora que hemos eliminado aquellos elementos que no nos interesan, podemos ver con qué etiquetas (`labels`) se han quedado los textos que sí forman parte del contenido principal del documento. Vemos que ahora tenemos principalmente `paragraph`, `section_header` y `title`, que son las etiquetas que nos interesan para el análisis.

## Analizar etiquetas

In [ ]:
labels = []
for item, level in doc.iterate_items():
    labels.append(item.label)

print(set(labels))

In [ ]:
for item, level in doc.iterate_items():
    if item.label == DocItemLabel.FOOTNOTE:
        print(item.self_ref, item.text)

In [ ]:
rich.print(doc.texts[824])

Si nos fijamos en el PDF original vemos que lo anterior está escrito en pequeño debajo de algunas tablas. No es estrictamente un pie de página, pero Docling lo ha etiquetado como `page_footer`, seguramente por el tamaño de la fuente.

In [ ]:
for item, level in doc.iterate_items():
    if item.label == DocItemLabel.CAPTION:
        print(item.self_ref, item.orig)

In [ ]:
rich.print(doc.texts[779])

In [ ]:
rich.print(doc.texts[784])

Esto es realidad no es ninguan caption, si no un más bien un `section_header` como si lo es el de `ANEXO III`, pero tampoco es algo grave.

In [ ]:
for item, level in doc.iterate_items():
    if item.label == DocItemLabel.PAGE_HEADER:
        print(item.self_ref, item.orig)

In [ ]:
for item, level in doc.iterate_items():
    if item.label == DocItemLabel.PAGE_FOOTER:
        print(item.self_ref, item.orig)

Vemos que hay algunos `page_header` y `page_footer` que no se han eliminado porque Docling no los había marcado como `furniture`. Esto se debe a que Docling no siempre es perfecto en la clasificación de los elementos, y a veces puede haber errores o casos límite. Por eso es importante revisar el resultado y hacer ajustes manuales si es necesario.

Para eliminarlos haríamos lo siguiente:

In [ ]:
total_texts = len(doc.texts)
print(f"Nº de textos en el documento: {total_texts}")

In [ ]:
items_to_delete = [
    item for item in doc.texts if item.label == DocItemLabel.PAGE_FOOTER or item.label == DocItemLabel.PAGE_HEADER
]
doc.delete_items(node_items=items_to_delete)

In [ ]:
total_texts_after_deletion = len(doc.texts)
print(f"Nº de textos en el documento: {total_texts_after_deletion}")

In [ ]:
for item, level in doc.iterate_items():
    if item.label == DocItemLabel.SECTION_HEADER:
        print(item.self_ref, item.orig)

In [ ]:
rich.print(doc.texts[2])

In [ ]:
for item, level in doc.iterate_items():
    if item.label == DocItemLabel.LIST_ITEM:
        print(item.self_ref, item.orig)

In [ ]:
# Si queremos guardar el JSON del documento
# limpio de elementos innecesarios
doc.save_as_json(PATH_OUTPUT / f"{pdf_files[0].stem}_clean.json")

PAGE_BREAK_PLACEHOLDER = "<!-- page_break -->"
md_content = doc.export_to_markdown(
    page_break_placeholder=PAGE_BREAK_PLACEHOLDER,
    include_annotations=True,
    mark_annotations=True,
)

# Guardamos el Markdown y el JSON resultante
out_path = PATH_OUTPUT / f"{pdf_files[0].stem}_clean.md"
out_path.write_text(md_content, encoding="utf-8")